<a href="https://colab.research.google.com/github/michaelsteven1299/proyecto_michael-/blob/main/src/01_consolidar.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Universidad Libre - Seccional Cali<br>Facultad de Ingeniería - Diplomado en Ciencia de Datos<br>(ↄ) Diego Fernando Marin, 2024

# 01_consolidar
Plantilla para el desarrollo del proyecto del diplomado de Ciencia de Datos, aplicando buenas prácticas.

---

Este cuaderno se enfoca en la integración de las distintas fuentes de datos en un formato cohesivo y estructurado. Aquí transformamos múltiples conjuntos de datos en una base unificada que servirá para los análisis posteriores.

**Propósito:** Crear una vista unificada y coherente de todos los datos recolectados, facilitando su posterior procesamiento y análisis.

**Tareas habituales:**
- Renombrar archivos
- Unión vertical de archivos complementarios (`union`)
- Combinar archivos (`joins`: inner, left, right, full outer)
- Estandarización inicial de formatos de columnas
- Verificación de consistencia en las uniones
- Validación de cardinalidad en las relaciones
- Gestión de duplicados producto de las uniones

In [1]:
from google.colab import drive
import os, pandas as pd

drive.mount('/content/drive')

RAW_PATH     = "/content/drive/MyDrive/proyecto_oro/data/raw/"
LANDING_PATH = "/content/drive/MyDrive/proyecto_oro/data/landing/"
os.makedirs(LANDING_PATH, exist_ok=True)

dfs = []

for archivo in os.listdir(RAW_PATH):
    if not archivo.endswith('.csv'):
        continue

    nombre = archivo.replace('.csv', '')
    df     = pd.read_csv(RAW_PATH + archivo, index_col=0, parse_dates=True)

    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)

    df.columns = [f"{nombre}_{col}" for col in df.columns]
    dfs.append(df)
    print(f"✓ {archivo}: {df.shape[0]:,} filas × {df.shape[1]} columnas")

df_landing = pd.concat(dfs, axis=1, join='outer')
df_landing = df_landing.sort_index()
df_landing = df_landing.ffill(limit=3)
df_landing = df_landing[df_landing.index.dayofweek < 5]
df_landing.index.name = 'DATE'
df_landing = df_landing.reset_index()
df_landing['DATE'] = df_landing['DATE'].astype(str)

ruta_salida = LANDING_PATH + "consolidado_oro_dxy.csv"
df_landing.to_csv(ruta_salida, index=False)

print(f"\n✅ Landing listo")
print(f"📊 {df_landing.shape[0]:,} filas × {df_landing.shape[1]} columnas")
print(f"📅 {df_landing['DATE'].min()} → {df_landing['DATE'].max()}")
df_landing.head()

Mounted at /content/drive
✓ wti_crudo.csv: 1,380 filas × 5 columnas
✓ plata_xagusd.csv: 1,380 filas × 5 columnas
✓ dxy.csv: 1,380 filas × 5 columnas
✓ oro_xauusd.csv: 1,380 filas × 5 columnas
✓ bono_10y.csv: 1,376 filas × 5 columnas
✓ vix.csv: 1,379 filas × 5 columnas
✓ eur_usd.csv: 1,428 filas × 5 columnas

✅ Landing listo
📊 1,429 filas × 36 columnas
📅 2021-01-01 → 2026-06-30


,DATE,wti_crudo_Close,wti_crudo_High,wti_crudo_Low,wti_crudo_Open,wti_crudo_Volume,plata_xagusd_Close,plata_xagusd_High,plata_xagusd_Low,plata_xagusd_Open,...,vix_Close,vix_High,vix_Low,vix_Open,vix_Volume,eur_usd_Close,eur_usd_High,eur_usd_Low,eur_usd_Open,eur_usd_Volume
0,2021-01-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,1.218027,1.221699,1.213499,1.217285,0.0
1,2021-01-04,47.619999,49.830002,47.180000,48.400002,528525.0,27.284000,27.284000,27.284000,27.284000,...,26.969999,29.190001,22.559999,23.040001,0.0,1.225070,1.230999,1.217137,1.224905,0.0
2,2021-01-05,49.930000,50.200001,47.240002,47.380001,643191.0,27.570999,27.570999,27.570999,27.570999,...,25.340000,28.600000,24.799999,26.940001,0.0,1.225160,1.229483,1.224995,1.225295,0.0
3,2021-01-06,50.630001,50.939999,49.480000,49.820000,509365.0,26.973000,27.424999,26.973000,27.424999,...,25.070000,26.770000,22.139999,25.480000,0.0,1.230027,1.235025,1.226693,1.229861,0.0
4,2021-01-07,50.830002,51.279999,50.389999,50.529999,369292.0,27.200001,27.200001,27.200001,27.200001,...,22.370001,23.910000,22.250000,23.670000,0.0,1.234111,1.234568,1.224665,1.233776,0.0
